# Предобработка датасетов для ЛР 1 по NLP

Датасеты:
1. **neo_task.csv** — классификация опасности астероидов (hazardous)
2. **winequality-red.csv + winequality-white.csv** — регрессия качества вина (quality)
3. *Текстовый датасет* — для Word2Vec (пока отсутствует, нужен "просьбы жильцов")

## 1. Настройка окружения

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Пути к данным
DATA_DIR = Path('datasets')
PROCESSED_DIR = Path('processed')
PROCESSED_DIR.mkdir(exist_ok=True)

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

## 2. Датасет neo_task.csv (классификация астероидов)

In [2]:
# Загрузка
neo_path = DATA_DIR / 'neo_task.csv'
neo = pd.read_csv(neo_path)

print(f"Размер: {neo.shape}")
print("\nИнформация:")
neo.info()
print("\nПервые 5 строк:")
neo.head()

Размер: (90836, 8)

Информация:
<class 'pandas.DataFrame'>
RangeIndex: 90836 entries, 0 to 90835
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  90266 non-null  float64
 1   name                90790 non-null  str    
 2   est_diameter_min    90836 non-null  float64
 3   est_diameter_max    90379 non-null  float64
 4   relative_velocity   90279 non-null  float64
 5   miss_distance       90836 non-null  float64
 6   absolute_magnitude  90152 non-null  float64
 7   hazardous           90836 non-null  bool   
dtypes: bool(1), float64(6), str(1)
memory usage: 4.9 MB

Первые 5 строк:


,id,name,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,absolute_magnitude,hazardous
0,3.5610e+06,(2011 GZ2),0.0160,0.0358,56014.0785,1.0243e+06,26.1,False
1,5.4017e+07,(2020 HT6),0.0305,0.0682,7864.3481,3.2682e+07,24.7,False
2,3.7466e+06,(2016 ED156),0.0555,0.1242,55257.5445,6.5386e+07,23.4,False
3,3.6331e+06,(2013 FD8),0.0193,0.0431,41531.4047,1.2608e+07,25.7,False
4,3.7421e+06,(2016 CW31),0.1395,0.3119,67639.3945,7.1306e+07,21.4,False


In [3]:
# Проверка пропусков
print("Пропуски по столбцам:")
neo.isna().sum()

Пропуски по столбцам:


id                    570
name                   46
est_diameter_min        0
est_diameter_max      457
relative_velocity     557
miss_distance           0
absolute_magnitude    684
hazardous               0
dtype: int64

In [4]:
# Обработка пропусков в id — заполнение по name или генерация
def fill_id_by_name(df):
    df = df.copy()
    
    # Приводим id к числовому типу
    df['id'] = pd.to_numeric(df['id'], errors='coerce')
    
    # Заполняем пропуски по существующим name
    names_with_id = df[df['id'].notna() & df['name'].notna()]
    name_to_id = names_with_id.groupby('name')['id'].first().to_dict()
    
    mask = df['id'].isna() & df['name'].isin(name_to_id)
    df.loc[mask, 'id'] = df.loc[mask, 'name'].map(name_to_id)
    
    # Генерируем новые id для безымянных
    missing_name = df.loc[df['id'].isna(), 'name'].dropna().unique()
    new_names = [n for n in missing_name if n not in name_to_id]
    
    if new_names:
        current_max = df['id'].max()
        if pd.isna(current_max):
            current_max = 0
        new_ids = range(int(current_max) + 1, int(current_max) + 1 + len(new_names))
        new_name_to_id = dict(zip(new_names, new_ids))
        
        mask_new = df['id'].isna() & df['name'].isin(new_names)
        df.loc[mask_new, 'id'] = df.loc[mask_new, 'name'].map(new_name_to_id)
    
    # Удаляем строки без name и id
    df = df.dropna(subset=['id'])
    
    return df

neo = fill_id_by_name(neo)
print(f"Размер после обработки id: {neo.shape}")
print(f"Пропусков в id: {neo['id'].isna().sum()}")

Размер после обработки id: (90836, 8)
Пропусков в id: 0


In [5]:
# Обработка пропусков в name — присваиваем имя по id
def fill_name_by_id(df):
    df = df.copy()
    mask = df['name'].isna()
    df.loc[mask, 'name'] = 'asteroid_' + df.loc[mask, 'id'].astype(int).astype(str)
    return df

neo = fill_name_by_id(neo)
print(f"Пропусков в name: {neo['name'].isna().sum()}")

Пропусков в name: 0


In [6]:
# Обработка пропусков в числовых столбцах по id
def fill_numeric_by_id(df, col):
    df = df.copy()
    before = df[col].isna().sum()
    
    # Заполняем по первому вхождению для каждого id
    valid = df[df[col].notna()]
    id_to_val = valid.groupby('id')[col].first().to_dict()
    
    mask = df[col].isna() & df['id'].isin(id_to_val)
    df.loc[mask, col] = df.loc[mask, 'id'].map(id_to_val)
    
    filled = before - df[col].isna().sum()
    print(f"{col}: заполнено {filled}, осталось {df[col].isna().sum()}")
    return df

for col in ['est_diameter_max', 'relative_velocity', 'absolute_magnitude']:
    neo = fill_numeric_by_id(neo, col)

est_diameter_max: заполнено 402, осталось 55
relative_velocity: заполнено 488, осталось 69
absolute_magnitude: заполнено 599, осталось 85


In [7]:
# Удаляем строки с оставшимися пропусками
neo = neo.dropna(subset=['est_diameter_max', 'relative_velocity', 'absolute_magnitude'])
print(f"Размер после удаления: {neo.shape}")
print("\nПропуски:")
neo.isna().sum()

Размер после удаления: (90627, 8)

Пропуски:


id                    0
name                  0
est_diameter_min      0
est_diameter_max      0
relative_velocity     0
miss_distance         0
absolute_magnitude    0
hazardous             0
dtype: int64

In [8]:
# Обработка hazardous — удаление одной строки с NaN и преобразование в int
neo = neo.dropna(subset=['hazardous'])
neo['hazardous'] = neo['hazardous'].map({False: 0, True: 1}).astype(int)
print(f"Пропусков в hazardous: {neo['hazardous'].isna().sum()}")
print(f"Тип: {neo['hazardous'].dtype}")
print(f"Распределение:\n{neo['hazardous'].value_counts()}")

Пропусков в hazardous: 0
Тип: int64
Распределение:
hazardous
0    81804
1     8823
Name: count, dtype: int64


In [9]:
# Оптимизация типов данных
neo['id'] = neo['id'].astype(int)
float_cols = ['est_diameter_min', 'est_diameter_max', 'relative_velocity', 
              'miss_distance', 'absolute_magnitude']
neo[float_cols] = neo[float_cols].astype('float32')

print("Типы данных:")
neo.dtypes

Типы данных:


id                      int64
name                      str
est_diameter_min      float32
est_diameter_max      float32
relative_velocity     float32
miss_distance         float32
absolute_magnitude    float32
hazardous               int64
dtype: object

In [10]:
# Сохранение предобработанного датасета
neo_processed_path = PROCESSED_DIR / 'neo_task_processed.csv'
neo.to_csv(neo_processed_path, index=False)
print(f"Сохранено: {neo_processed_path}")
print(f"Размер: {neo.shape}")

Сохранено: processed\neo_task_processed.csv
Размер: (90627, 8)


## 3. Датасеты winequality (регрессия качества вина)

In [11]:
# Загрузка
red_path = DATA_DIR / 'winequality-red.csv'
white_path = DATA_DIR / 'winequality-white.csv'

red = pd.read_csv(red_path, sep=';')
white = pd.read_csv(white_path, sep=';')

print(f"Красное вино: {red.shape}")
print(f"Белое вино: {white.shape}")

Красное вино: (1599, 12)
Белое вино: (4898, 12)


In [12]:
# Проверка пропусков
print("Пропуски в красном:")
print(red.isna().sum())
print("\nПропуски в белом:")
print(white.isna().sum())

Пропуски в красном:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

Пропуски в белом:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64


In [13]:
# Объединение датасетов
red['wine_type'] = 'red'
white['wine_type'] = 'white'

wine = pd.concat([red, white], ignore_index=True)
print(f"Объединённый датасет: {wine.shape}")

Объединённый датасет: (6497, 13)


In [14]:
# Проверка дубликатов
duplicates = wine.duplicated().sum()
print(f"Дубликатов: {duplicates}")
if duplicates > 0:
    wine = wine.drop_duplicates()
    print(f"После удаления: {wine.shape}")

Дубликатов: 1177
После удаления: (5320, 13)


In [15]:
# Преобразование типов
num_cols = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
            'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
            'density', 'pH', 'sulphates', 'alcohol', 'quality']

wine[num_cols] = wine[num_cols].astype('float32')
wine['wine_type'] = wine['wine_type'].map({'red': 0, 'white': 1}).astype(int)

print("Типы данных:")
wine.dtypes

Типы данных:


fixed acidity           float32
volatile acidity        float32
citric acid             float32
residual sugar          float32
chlorides               float32
free sulfur dioxide     float32
total sulfur dioxide    float32
density                 float32
pH                      float32
sulphates               float32
alcohol                 float32
quality                 float32
wine_type                 int64
dtype: object

In [16]:
# Сохранение предобработанного датасета
wine_processed_path = PROCESSED_DIR / 'winequality_processed.csv'
wine.to_csv(wine_processed_path, index=False)
print(f"Сохранено: {wine_processed_path}")
print(f"Размер: {wine.shape}")

Сохранено: processed\winequality_processed.csv
Размер: (5320, 13)


## 4. Анализ и очистка выбросов (опционально для обучения)

In [17]:
# Функция для определения границ выбросов (3*IQR)
def get_outlier_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 3 * iqr
    upper = q3 + 3 * iqr
    return lower, upper

# Анализ выбросов для neo
print("Выбросы в neo_task:")
for col in float_cols:
    lower, upper = get_outlier_bounds(neo[col])
    outliers = ((neo[col] < lower) | (neo[col] > upper)).sum()
    print(f"{col}: {outliers} выбросов (границы: [{lower:.2f}, {upper:.2f}])")

Выбросы в neo_task:
est_diameter_min: 3876 выбросов (границы: [-0.35, 0.52])
est_diameter_max: 3876 выбросов (границы: [-0.79, 1.15])
relative_velocity: 32 выбросов (границы: [-74334.34, 165915.52])
miss_distance: 0 выбросов (границы: [-100636226.00, 174484956.00])
absolute_magnitude: 0 выбросов (границы: [8.26, 38.78])


In [18]:
# Анализ выбросов для wine
print("\nВыбросы в winequality:")
for col in num_cols:
    lower, upper = get_outlier_bounds(wine[col])
    outliers = ((wine[col] < lower) | (wine[col] > upper)).sum()
    print(f"{col}: {outliers} выбросов (границы: [{lower:.2f}, {upper:.2f}])")


Выбросы в winequality:
fixed acidity: 83 выбросов (границы: [2.50, 11.60])
volatile acidity: 36 выбросов (границы: [-0.31, 0.95])
citric acid: 10 выбросов (границы: [-0.24, 0.88])
residual sugar: 3 выбросов (границы: [-15.30, 24.60])
chlorides: 119 выбросов (границы: [-0.05, 0.15])
free sulfur dioxide: 8 выбросов (границы: [-59.00, 116.00])
total sulfur dioxide: 1 выбросов (границы: [-163.75, 391.00])
density: 1 выбросов (границы: [0.98, 1.01])
pH: 2 выбросов (границы: [2.45, 3.99])
sulphates: 30 выбросов (границы: [-0.08, 1.11])
alcohol: 0 выбросов (границы: [3.80, 17.10])
quality: 0 выбросов (границы: [2.00, 9.00])


## 5. Итоговая сводка

In [19]:
print("=== Сводка по предобработанным датасетам ===\n")

print("neo_task_processed.csv:")
print(f"  Размер: {neo.shape}")
print(f"  Целевая переменная: hazardous (бинарная классификация)")
print(f"  Баланс классов: {neo['hazardous'].value_counts().to_dict()}")

print("\nwinequality_processed.csv:")
print(f"  Размер: {wine.shape}")
print(f"  Целевая переменная: quality (регрессия, 3-9 баллов)")
print(f"  Среднее качество: {wine['quality'].mean():.2f}")
print(f"  Распределение типов вина: {wine['wine_type'].value_counts().to_dict()}")

print("\nПредобработанные файлы сохранены в:", PROCESSED_DIR.absolute())

=== Сводка по предобработанным датасетам ===

neo_task_processed.csv:
  Размер: (90627, 8)
  Целевая переменная: hazardous (бинарная классификация)
  Баланс классов: {0: 81804, 1: 8823}

winequality_processed.csv:
  Размер: (5320, 13)
  Целевая переменная: quality (регрессия, 3-9 баллов)
  Среднее качество: 5.80
  Распределение типов вина: {1: 3961, 0: 1359}

Предобработанные файлы сохранены в: C:\Users\trape\OneDrive\Desktop\Studies 2026\NLP\solution\lab1\processed


## 6. Текстовый датасет для Word2Vec

**Требуется:** датасет "просьбы жильцов" или любой другой текстовый датасет для классификации.

Когда будет получен, нужно:
1. Загрузить текстовый столбец
2. Выполнить предобработку: нижний регистр, удаление спецсимволов, токенизация
3. Удалить стоп-слова (опционально)
4. Сохранить предобработанный текст
5. Для Word2Vec использовать лемматизацию (ru_core_news_sm для русского текста)